In [ ]:
from micrograd.engine import Value


a = Value(-5.2)
b = Value(2.2009)
c = a + b
d = a * b + b**3
c += c + 1
c += 1 + c + (-a)
d += d * 2 + (b + a).relu()
d += 3 * d + (b - a).relu()
e = c - d
f = e**2
g = f / 2.0
g += 10.0 / f
print(f'{g.data:.4f}') # prints 24.7041, the outcome of this forward pass
g.backward() 
print(f'{a.grad:.4f}') # prints 138.8338, i.e. the numerical value of dg/da
print(f'{b.grad:.4f}') # prints 645.5773, i.e. the numerical value of dg/db

In [ ]:
class value:
    def __init__(self,data):
        self.data =data

    def __repr__(self):
        return f"value(data={self.data})"

In [ ]:
class value:
    def __init__(self,data):
        self.data =data

    def __repr__(self):
        return f"value(data={self.data})"

    def __add__(self,other):
        out = value(self.data + other.data)
        return out
    def __mul__(self,other):
        out =value(self.data * other.data)
        return out

a=value(2.0)
b=value(-3.0)
c=10.0
a+b

In [ ]:
class value:
    def __init__(self,data,_children=(),_op=''):
        self.data =data
        self._prev= set(_children)
        self.op=_op
    def __repr__(self):
        return f"value(data={self.data})"

    def __add__(self,other):
        out = value(self.data + other.data, (self,other), '+')
        return out
    def __mul__(self,other):
        out =value(self.data * other.data,(self,other), '*')
        return out

a=value(2.0)
b=value(-3.0)
c=value(10.0)
d=a*b + c
d

In [ ]:
class value:
    def __init__(self,data,_children=(),_op=''):
        self.data =data
        self._prev= set(_children)
        self._op=_op

    def __repr__(self):
        return f"value(data={self.data})"

    def __add__(self,other,_op='+'):
        out = value(self.data + other.data, (self,other), _op)
        return out
    def __mul__(self,other,_op='*'):
        out =value(self.data * other.data,(self,other), _op)
        return out

a=value(2.0)
b=value(-3.0)
e=a*b
c=value(10.0)
d=a*b + c
d

In [ ]:
e._detail=np.array([e._op, e._prev])
d._detail=np.array([d._op, d._prev])
e._detail,d._detail

# this is brute way to represent in array form, but we can use graphviz to represent the graph in a better way as:

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

In [ ]:
from graphviz import Digraph

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir':'LR'})
    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name=uid,label="{ %s | data %.4f | grad %.4f }" % (n._op, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=uid + n._op, label=n._op)
            dot.edge(uid + n._op, uid)
    for n1,n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    return dot

In [ ]:
class value:
    def __init__(self, data, _children=(),_op='', label=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"value(data={self.data})"

    def __add__(self, other):
        out = value(self.data + other.data, (self, other), '+')
        return out

    def __mul__(self, other):   
        out = value(self.data * other.data, (self,other), '*')
        return out

a=value(2.0, label='a')
b=value(-3.0, label='b')
c=value(10.0, label='c')
e=a*b; e.label='e'
d=e+c; d.label='d'
d

In [ ]:
class value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data =data
        self.grad=0.0
        self._prev= set(_children)
        self._op=_op
        self.label=label
        self.grad=0

    def __repr__(self):
        return f"value(data={self.data})"

    def __add__(self,other,_op='+'):
        out = value(self.data + other.data, (self,other), _op)
        return out
    def __mul__(self,other,_op='*'):
        out =value(self.data * other.data,(self,other), _op)
        return out

a=value(2.0, label='a')
b=value(-3.0, label='b')
c=value(10.0, label='c')
e=a*b; e.label='e'
d=e + c; d.label='d'
f=value(-2.0, label='f')
L= d * f; L.label='L'
L

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.plot(np.arange(-5,5,0.1), np.tanh(np.arange(-5,5,0.1)));plt.grid()

In [ ]:
import math


class value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data =data
        self.grad=0.0
        self._prev= set(_children)
        self._op=_op
        self.label=label
        self.grad=0

    def __repr__(self):
        return f"value(data={self.data})"

    def __add__(self,other,_op='+'):
        out = value(self.data + other.data, (self,other), _op)
        return out
    def __mul__(self,other,_op='*'):
        out =value(self.data * other.data,(self,other), _op)
        return out
    def tanh(self):
        x=self.data
        t= (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out=value(t,(self,), 'tanh')
        return out


a=value(2.0, label='a')
b=value(-3.0, label='b')
c=value(10.0, label='c')
e=a*b; e.label='e'
d=e + c; d.label='d'
f=value(-2.0, label='f')
L= d * f; L.label='L'
L

In [ ]:
#taking inputs as: x1, x2 and weights:w1, w2
x1 = value(-4.0, label='x1')
x2 = value(2.0, label='x2')

w1 = value(-3.0, label='w1')
w2 = value(1.0, label='w2') 

b = value(0.5, label='b')  #biad of our neuron

x1w1 = x1 * w1; x1w1.label='x1w1'
x2w2 = x2 * w2; x2w2.label='x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label='x1*w1+x2*w2'

n = x1w1x2w2 + b; n.label='cellbody(n)' # our neuron's cell body, which sums up the inputs and weights and bias

#o = n.np.tanh(); o.label='o' # our neuron's output after applying the activation fucntion tanh to the cell body output

In [ ]:
draw_dot(o) #draw the graph of our neuron

In [ ]:
o.grad=1

In [ ]:
# o=tanh(n) -> do/dn=1-tanh^2(n)= 1-o^2
n.grad= 1 - o.data**2

In [ ]:
#pretty obvious that manual calculation of gradients(for backpropagation)
#is pretty dull way, so there is the necessity to automat it.
#this is carried out within the class, by implementing the backward function.
#backward function is implemented witihin each mathematical operation function inside the class(as per our example: +,*,tanh)
#calling the backward fucntion in right order will calculate the grad of each nodes in the graph

In [ ]:
import math
from sympy import Lambda


class value:
    def __init__(self,data,_children=(),_op='',label=''):
        self.data =data
        self.grad=0.0
        self._prev= set(_children)
        self._op=_op
        self.label=label
        self.grad=0.0
        self._backward= lambda: None

    def __repr__(self):
        return f"value(data={self.data})"

    def __add__(self,other):
        out = value(self.data + other.data, (self,other), '+')

        def _backward():
            self.grad= 1.0*out.grad
            other.grad= 1.0*out.grad 
        out._backward=_backward

        return out
    def __mul__(self,other):
        out =value(self.data * other.data,(self,other), '*')

        def _backward():
            self.grad= other.data * out.grad
            other.grad= self.data * out.grad
        out._backward=_backward

        return out
    def tanh(self):
        x=self.data
        t= (math.exp(2*x)-1)/(math.exp(2*x)+1)
        out=value(t,(self,), 'tanh')

        def _backward():
            self.grad= 1 - out.data**2
        out._backward = _backward

        return out


a=value(2.0, label='a')
b=value(-3.0, label='b')
c=value(10.0, label='c')
e=a*b; e.label='e'
d=e + c; d.label='d'
f=value(-2.0, label='f')
L= d * f; L.label='L'
L

In [ ]:
o.grad=1.0
o._backward()
value(n.grad)
n._backward()
value(x1w1x2w2.grad)
value(b.grad)
x1w1x2w2._backward()
value(x1w1.grad)
value(x2w2.grad)
x1w1._backward()
x2w2._backward()
x1.grad
x2.grad
w1.grad
w2.grad

In [ ]:
#using toplogical sort to transverse the graph in the rifht order for backpropagation.
#this is done by using a depth first search algorithm as:

topo=[]
visited =set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)
build_topo(o)
topo

In [ ]:
#this is the automation phase

o.grad=1.0

o._backward()
print(n.grad)

n._backward()
print(x1w1x2w2.grad)
print(b.grad)
print(x1w1x2w2.grad)

x1w1x2w2._backward()
print(x1w1.grad)
print(x2w2.grad)

x1w1._backward()
print(x1.grad)
print(x2.grad)

x2w2._backward()
print(w1.grad)
print(w2.grad)